In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

import lightgbm as lgb
import xgboost as xgb
import optuna

/Users/andyzhu/PycharmProjects/Kaggle_Titanic/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv("train.csv", index_col="PassengerId")
test = pd.read_csv("test.csv", index_col="PassengerId")

In [26]:
train.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Deck', 'Title',
       'FamilySize', 'IsAlone', 'FamilyCategory', 'Log_Fare', 'IsChild',
       'TicketFrequency'],
      dtype='str')

In [4]:
train.isnull().sum().sort_values(ascending=False).head(10)

Survived      0
Pclass        0
Sex           0
Age           0
Fare          0
Embarked      0
Deck          0
Title         0
FamilySize    0
IsAlone       0
dtype: int64

In [5]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 1 to 891
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Survived         891 non-null    int64  
 1   Pclass           891 non-null    int64  
 2   Sex              891 non-null    str    
 3   Age              891 non-null    int64  
 4   Fare             891 non-null    float64
 5   Embarked         891 non-null    str    
 6   Deck             891 non-null    str    
 7   Title            891 non-null    str    
 8   FamilySize       891 non-null    int64  
 9   IsAlone          891 non-null    int64  
 10  FamilyCategory   891 non-null    str    
 11  Log_Fare         891 non-null    float64
 12  IsChild          891 non-null    int64  
 13  TicketFrequency  891 non-null    int64  
dtypes: float64(2), int64(7), str(5)
memory usage: 97.6 KB


In [6]:
#Baseline Logistic Regression

In [7]:
X_logistic = train.drop(columns=["Survived", "Fare", "FamilySize"])
X_logistic_test = test.drop(columns=["Fare", "FamilySize"])
y = train["Survived"]

In [8]:
X_logistic.columns

Index(['Pclass', 'Sex', 'Age', 'Embarked', 'Deck', 'Title', 'IsAlone',
       'FamilyCategory', 'Log_Fare', 'IsChild', 'TicketFrequency'],
      dtype='str')

In [18]:
X_logistic_test[X_logistic_test.Log_Fare.isnull()]

,Pclass,Sex,Age,Embarked,Deck,Title,IsAlone,FamilyCategory,Log_Fare,IsChild,TicketFreqCategory
PassengerId,,,,,,,,,,,
1044,3,male,60,S,0,Mr,1,Alone,NaN,0,Large


In [10]:
#Denn die TicketFrequency ist U Förmig mit Survived Korreliert
def ticket_freq_category(freq):
    if freq == 1:
        return 'Solo'
    elif freq <= 4:
        return 'Small'
    else:
        return 'Large'

X_logistic['TicketFreqCategory'] = X_logistic['TicketFrequency'].apply(ticket_freq_category)
X_logistic = X_logistic.drop(columns=['TicketFrequency'])

X_logistic_test['TicketFreqCategory'] = X_logistic_test['TicketFrequency'].apply(ticket_freq_category)
X_logistic_test = X_logistic_test.drop(columns=['TicketFrequency'])

In [11]:
numeric_cols_logistic = X_logistic.select_dtypes(include="number").columns
categorical_cols_logistic = X_logistic.select_dtypes(include="str").columns
categorical_cols_logistic

Index(['Sex', 'Embarked', 'Deck', 'Title', 'FamilyCategory',
       'TicketFreqCategory'],
      dtype='str')

In [12]:
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_preprocessor = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric_preprocessor", numeric_preprocessor, numeric_cols_logistic),
    ("categorical_preprocessor", categorical_preprocessor, categorical_cols_logistic),
])
Baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])
Baseline.fit(X_logistic, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['Pclass','Sex','Age',...,'Log_Fare','IsChild','TicketFreqCategory']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_preprocessor', ...), ('categorical_preprocessor', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By spec

In [29]:
Fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [30]:
cross_val_score(Baseline, X_logistic, y, cv=Fold, scoring="accuracy").mean()

np.float64(0.8293766869625259)

In [15]:
def submission(model, X_test):
    y_pred = model.predict(X_test)

    Predictions = pd.DataFrame({
        "PassengerId": X_test.index,
        "Survived": y_pred,
    })
    return Predictions

In [16]:
submission_baseline = submission(Baseline, X_logistic_test)
submission_baseline.to_csv("submission_baseline.csv", index=False)

In [17]:
#Mit Boosting

In [21]:
X_tree = train.drop(columns=["Survived", "Log_Fare"])
X_tree_test = test.drop(columns=["Log_Fare"])

In [25]:
X_tree.columns

Index(['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Deck', 'Title',
       'FamilySize', 'IsAlone', 'FamilyCategory', 'IsChild',
       'TicketFrequency'],
      dtype='str')

In [24]:
categorical_cols_tree = X_tree.select_dtypes(include="str").columns
categorical_cols_tree

Index(['Sex', 'Embarked', 'Deck', 'Title', 'FamilyCategory'], dtype='str')

In [27]:
for col in categorical_cols_tree:
    X_tree[col] = X_tree[col].astype('category')
    X_tree_test[col] = X_tree_test[col].astype('category')

In [44]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 4, 32),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X_tree, y):
        X_tr, X_val = X_tree.iloc[train_idx], X_tree.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**params, random_state=42)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )

        preds = model.predict(X_val)
        scores.append(accuracy_score(y_val, preds))

    return np.mean(scores)

In [46]:
study = optuna.create_study(
    direction="maximize",
    study_name="titanic_lgbm_v2",
    storage='sqlite:///optuna_lgbm.db',
    load_if_exists=True,
)
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-06-21 13:10:58,121] Using an existing study with name 'titanic_lgbm_v2' instead of creating a new one.
[I 2026-06-21 13:10:59,308] Trial 42 finished with value: 0.8473542150524136 and parameters: {'n_estimators': 366, 'learning_rate': 0.03566673249782356, 'num_leaves': 26, 'max_depth': 5, 'min_child_samples': 60, 'subsample': 0.9704869910119537, 'colsample_bytree': 0.9161135491339499, 'reg_alpha': 1.6410080215901946e-06, 'reg_lambda': 8.060630501790477e-07}. Best is trial 25 with value: 0.8518423200050217.
[I 2026-06-21 13:11:00,185] Trial 43 finished with value: 0.8406000878789781 and parameters: {'n_estimators': 285, 'learning_rate': 0.0321835440628769, 'num_leaves': 26, 'max_depth': 5, 'min_child_samples': 51, 'subsample': 0.9393666561571714, 'colsample_bytree': 0.9412147205917123, 'reg_alpha': 0.0005782377051011999, 'reg_lambda': 4.6958829734898475e-07}. Best is trial 25 with value: 0.8518423200050217.
[I 2026-06-21 13:11:01,301] Trial 44 finished with value: 0.841742514594

Best params: {'n_estimators': 330, 'learning_rate': 0.12765404839393585, 'num_leaves': 12, 'max_depth': 6, 'min_child_samples': 64, 'subsample': 0.9102318778617753, 'colsample_bytree': 0.9014137360044202, 'reg_alpha': 1.0222248734906468e-08, 'reg_lambda': 0.0012156594146630277}
Best score: 0.8540895110162575


In [47]:
study.trials_dataframe().sort_values('value', ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_min_child_samples,params_n_estimators,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,state
52,52,0.854090,2026-06-21 13:11:08.297306,2026-06-21 13:11:08.908837,0 days 00:00:00.611531,0.901414,0.127654,6,64,330,12,1.022225e-08,0.001216,0.910232,COMPLETE
61,61,0.852966,2026-06-21 13:11:15.534601,2026-06-21 13:11:16.331454,0 days 00:00:00.796853,0.999269,0.134102,6,64,386,23,1.537853e+00,0.000017,0.946938,COMPLETE
85,85,0.852960,2026-06-21 13:11:31.669061,2026-06-21 13:11:32.291011,0 days 00:00:00.621950,0.911133,0.113741,6,60,362,18,1.018979e-03,0.000057,0.928520,COMPLETE
77,77,0.851861,2026-06-21 13:11:25.789339,2026-06-21 13:11:26.465190,0 days 00:00:00.675851,0.887857,0.108592,5,59,318,19,1.436701e-02,0.000269,0.948643,COMPLETE
25,25,0.851842,2026-06-21 13:10:37.949550,2026-06-21 13:10:38.906526,0 days 00:00:00.956976,0.951577,0.065880,5,63,333,28,4.254873e-08,0.018020,0.891182,COMPLETE
62,62,0.851836,2026-06-21 13:11:16.335657,2026-06-21 13:11:17.009617,0 days 00:00:00.673960,0.997076,0.132374,6,64,387,24,1.702247e+00,0.000141,0.945981,COMPLETE
32,32,0.850719,2026-06-21 13:10:44.357287,2026-06-21 13:10:45.127294,0 days 00:00:00.770007,0.916229,0.118380,6,65,324,25,3.995454e-08,0.001353,0.903931,COMPLETE
76,76,0.850719,2026-06-21 13:11:25.071092,2026-06-21 13:11:25.784992,0 days 00:00:00.713900,0.887784,0.107263,6,58,383,23,2.313499e-02,0.000320,0.905031,COMPLETE
58,58,0.849601,2026-06-21 13:11:12.785669,2026-06-21 13:11:13.531711,0 days 00:00:00.746042,0.905275,0.133407,6,63,368,25,1.483695e+00,0.000107,0.948968,COMPLETE
79,79,0.849576,2026-06-21 13:11:27.120799,2026-06-21 13:11:27.853963,0 days 00:00:00.733164,0.865123,0.103510,6,58,372,19,2.910984e-02,0.000121,0.829577,COMPLETE


In [50]:
best_model_lgbm = lgb.LGBMClassifier(**study.best_params)
best_model_lgbm.fit(X_tree, y)

,num_leaves,12
,max_depth,6
,learning_rate,0.12765404839393585
,n_estimators,330
,min_child_samples,64
,subsample,0.9102318778617753
,colsample_bytree,0.9014137360044202
,reg_alpha,1.0222248734906468e-08
,reg_lambda,0.0012156594146630277
,boosting_type,'gbdt'
,subsample_for_bin,200000


In [49]:
cross_val_score(best_model_lgbm, X_tree, y, scoring="accuracy").mean()

np.float64(0.8383842822170611)

In [51]:
submission_lgbm = submission(best_model_lgbm, X_tree_test)
submission_lgbm.to_csv("submission_lgbm_2.csv", index=False)